**Итоговый проект по Python**

In [6]:
# endpoint (URL-адрес), на который можно отправить запрос
api_url = "https://b2b.itresume.ru/api/statistics"

# задаем начальную и конечную даты
start = '2023-04-01 12:46:47.860798'
end = '2023-04-30 09:16:30.117858'

# параметры для API
params = {'client': 'Skillfactory',
          'client_key': 'M2MGWS',
          'start': start,
          'end': end}

In [7]:
# параметры БД

host = "localhost"
port = 5432
database = "postgres"
user = "postgres"
password = "TanyaMich8404"

In [8]:
# ETL-процесс № 1 (Extract, Transform, Load) для получения данных из API, обработки и загрузки в PostgreSQL

import os
from logging.handlers import TimedRotatingFileHandler
import requests
from requests.exceptions import RequestException, Timeout, ConnectionError, HTTPError
import json
import time
import ast
from datetime import datetime, timedelta
import logging
import psycopg2
import re
from psycopg2 import sql
import psycopg2.extras
import glob

def setup_logger(name='etl_logger', log_dir='logs', level=logging.INFO):

    """Настройка логирования с ротацией файлов и очисткой старых логов
    args:
        name: имя логгера
        log_dir: директория для хранения логов
        level: уровень логирования
    """

    # Полный сброс системы логирования перед запуском
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    logger = logging.getLogger(name)
    logger.setLevel(level)

    if logger.hasHandlers():
        logger.handlers.clear()

    os.makedirs(log_dir, exist_ok=True)

    formatter = logging.Formatter(
        fmt='%(asctime)s | %(levelname)s | %(name)s | %(funcName)s | %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )

    current_date = datetime.now().strftime('%Y-%m-%d')
    log_file = os.path.join(log_dir, f'pipeline_{current_date}.log')

    file_handler = TimedRotatingFileHandler(
        log_file, when='midnight', interval=1, backupCount=3, encoding='utf-8'
    )
    file_handler.setFormatter(formatter)
    file_handler.setLevel(logging.INFO)

    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    console_handler.setLevel(logging.INFO)

    logger.addHandler(file_handler)
    logger.addHandler(console_handler)
    logger.propagate = False

    # Внутренняя очистка старых логов
    cutoff_date = datetime.now() - timedelta(days=3)
    log_files = glob.glob(os.path.join(log_dir, 'pipeline_*.log'))

    for log_file_path in log_files:
        try:
            filename = os.path.basename(log_file_path)
            file_date_str = filename.replace('pipeline_', '').replace('.log', '')
            file_date = datetime.strptime(file_date_str, '%Y-%m-%d')

            if file_date < cutoff_date:
                os.remove(log_file_path)
        except (ValueError, OSError):
            pass  # Игнорируем ошибки при удалении

    return logger

#2. Обращение к API для получения данных


def get_data_from_api(api_url, params=None, timeout=120, logger=None):

    """Получение данных из API с обработкой ошибок и логированием
    args:
        api_url: URL-адрес API
        params: параметры запроса
        timeout: таймаут запроса
        logger: экземпляр логгера
    """

    if logger is None:
        logger = logging.getLogger(__name__)

    start_time = time.time()
    logger.info(f"Начало запроса к API {api_url}, параметры {params}")

    try:
        response = requests.get(api_url, params=params, timeout=timeout)

        duration = time.time() - start_time

        if response.status_code == 429:
            logger.warning(f"429 (rate limit) за {duration:.2f} с")
            return None

        if response.status_code >= 400:
            logger.warning(
                f"HTTP {response.status_code} за {duration:.2f} с: "
                f"{response.reason}"
            )
            return None

        try:
            data = response.json()

            logger.info(
                f"Успешный ответ от API. "
                f"Время: {duration:.2f} с"
            )

            return data

        except json.JSONDecodeError as err:
            logger.error(f"Ошибка парсинга JSON: {err}")
            return None

    except Timeout:
        logger.error(f"Таймаут запроса к {api_url} (> {timeout} сек)")
        return None

    except ConnectionError:
        logger.error(f"Ошибка подключения к {api_url}")
        return None

    except HTTPError as err:
        logger.error(f"HTTP ошибка: {err}")
        return None

    except RequestException as err:
        logger.error(f"Ошибка запроса: {err}")
        return None

    except Exception as err:
        logger.error(f"Неожиданная ошибка: {err}")
        return None


#3. Обработка сырых данных и подготовка к загрузке в базу данных

def valid_url(url):
    return isinstance(url, str) and url.startswith('https://')


def data_processing(raw_data, logger=None):

    """Обработка сырых данных из API, валидация и подготовка к загрузке
    args:
        raw_data: список словарей с сырыми данными
        logger: экземпляр логгера
    """

    start_time = time.time()

    if logger is None:
        logger = logging.getLogger(__name__)

    logger.info(f"Начало обработки {len(raw_data)} записей")

    processed_data = []
    skipped_records = 0

    for record in raw_data:
        try:
            lti_user_id = record.get('lti_user_id')

            passback_str = record.get('passback_params')
            if not passback_str:
                skipped_records += 1
                continue

            passback_params = ast.literal_eval(passback_str)

            oauth_consumer_key = passback_params.get('oauth_consumer_key')
            lis_result_sourcedid = passback_params.get('lis_result_sourcedid')
            lis_outcome_service_url = passback_params.get('lis_outcome_service_url')

            # обязательные поля
            if not all([lis_result_sourcedid, lis_outcome_service_url]):
                skipped_records += 1
                continue

            if not valid_url(lis_outcome_service_url):
                skipped_records += 1
                continue

            # остальные поля
            is_correct = record.get('is_correct')
            attempt_type = record.get('attempt_type')
            created_at = record.get('created_at')

            # нормализация boolean
            if is_correct in (1, '1', True, 'true', 'True'):
                is_correct = True
            elif is_correct in (0, '0', False, 'false', 'False'):
                is_correct = False
            else:
                is_correct = None

            # нормализация даты
            if isinstance(created_at, str):
                try:
                    created_at = datetime.fromisoformat(created_at)
                except ValueError:
                    skipped_records += 1
                    continue

            processed_data.append({
                'lti_user_id': lti_user_id,
                'oauth_consumer_key': oauth_consumer_key,
                'lis_result_sourcedid': lis_result_sourcedid,
                'lis_outcome_service_url': lis_outcome_service_url,
                'is_correct': is_correct,
                'attempt_type': attempt_type,
                'created_at': created_at
            })

        except Exception as e:
            logger.error(f"Ошибка обработки записи: {e}")
            skipped_records += 1

    duration = time.time() - start_time

    logger.info(
        f"Готово! Всего: {len(raw_data)} | "
        f"Обработано: {len(processed_data)} | "
        f"Пропущено: {skipped_records} | "
        f"Время: {duration:.2f} сек"
    )

    return processed_data


#4. Загрузка обработанных данных в Postgres

def load_data_to_postgres(processed_data, database, user, password, host, port, table_name='api_data', logger=None):

    """Загрузка обработанных данных в PostgreSQL с безопасным созданием таблицы и вставкой данных
    args:
        processed_data: список словарей с обработанными данными
        database: имя базы данных
        user: имя пользователя
        password: пароль
        host: хост базы данных
        port: порт базы данных
        table_name: имя таблицы
        logger: экземпляр логгера
    """

    start_time = time.time()

    if logger is None:
        logger = logging.getLogger(__name__)

    logger.info("Начало загрузки данных в PostgreSQL")

    # Валидация имени таблицы — только буквы, цифры, подчёркивания
    if not re.match(r'^[a-zA-Z_][a-zA-Z0-9_]*$', table_name):
        raise ValueError(f"Недопустимое имя таблицы: {table_name}")

    if not processed_data:
        print("Нет данных для загрузки!")
        logger.warning("Нет данных для загрузки!")
        return None

    conn = None
    try:
        with psycopg2.connect(
                database=database,
                user=user,
                password=password,
                host=host,
                port=port
        ) as conn:

            with conn.cursor() as cursor:
                # Безопасное создание таблицы
                create_table_query = sql.SQL(
                    "CREATE TABLE IF NOT EXISTS {} ("
            "lti_user_id VARCHAR(255),"
            "oauth_consumer_key VARCHAR(255),"
            "lis_result_sourcedid TEXT NOT NULL,"
            "lis_outcome_service_url TEXT NOT NULL,"
            "is_correct BOOLEAN,"
            "attempt_type VARCHAR(50),"
            "created_at TIMESTAMP"
            ")"
                ).format(sql.Identifier(table_name))
                cursor.execute(create_table_query)

                # Шаблон для execute_values — используем %s один раз как плейсхолдер для всей строки данных
                template = sql.SQL("(%s, %s, %s, %s, %s, %s, %s)")

                # Основной запрос с плейсхолдером %(data)s для списка строк
                insert_query = sql.SQL(
            "INSERT INTO {} ("
            "lti_user_id, oauth_consumer_key,"
            "lis_result_sourcedid,"
            "lis_outcome_service_url,"
            "is_correct,"
            "attempt_type,"
            "created_at"
            ") VALUES %s"
                ).format(sql.Identifier(table_name))

                # Подготовка данных
                values = [
                    (
                        record.get('lti_user_id'),
                        record.get('oauth_consumer_key'),
                        record.get('lis_result_sourcedid'),
                        record.get('lis_outcome_service_url'),
                        record.get('is_correct'),
                        record.get('attempt_type'),
                        record.get('created_at')
                    )
                for record in processed_data
            ]

                # Использование execute_values с шаблоном
                psycopg2.extras.execute_values(
                    cursor,
                    insert_query,
                    values,
                    template=template,
                    page_size=1000
        )
                conn.commit()
                logger.info(f"Успешно загружено {len(processed_data)} записей в таблицу {table_name}.")

    except psycopg2.Error as e:
        logger.error(f"Ошибка PostgreSQL при работе с таблицей {table_name}: {e}")
        if conn:
            conn.rollback()
        raise
    except Exception as e:
        logger.error(f"Неожиданная ошибка при загрузке данных: {e}")
        raise
    finally:
        if conn:
            end_time = time.time()
            duration = end_time - start_time
            logger.info(f"Соединение с PostgreSQL закрыто! Время обработки: {duration:.2f} с.")



In [9]:
# Запуск ETL-процесс № 1: получение данных из API → обработка → загрузка в PostgreSQL

def run_etl(api_url, params, database, user, password, host, port, table_name='api_data', logger=None):

    """Запуск полного ETL-процесса: от получения данных из API до загрузки в PostgreSQL
    args:
        api_url: URL-адрес API
        params: параметры запроса
        database: имя базы данных
        user: имя пользователя
        password: пароль
        host: хост базы данных
        port: порт базы данных
        table_name: имя таблицы
        logger: экземпляр логгера
    """

    logger.info("Запуск полного ETL-процесса")

    try:
        # Шаг 1: Получение данных из API
        logger.info("Получение данных из API...")
        raw_data = get_data_from_api(api_url, params=params, logger=logger)

        if not raw_data:
            logger.warning("Не удалось получить данные из API.")
            return False

        # Шаг 2: Обработка данных
        logger.info("Обработка полученных данных...")
        processed_data = data_processing(raw_data, logger=logger)

        if not processed_data:
            logger.warning("Нет обработанных данных для загрузки.")
            return False

        # Шаг 3: Загрузка данных в PostgreSQL
        logger.info("Загрузка данных в PostgreSQL...")
        print("Попытка загрузить данные в PostgreSQL. Пожалуйста, убедитесь, что ваш сервер PostgreSQL запущен")

        load_data_to_postgres(
            processed_data=processed_data,
            database=database,
            user=user,
            password=password,
            host=host,
            port=port,
            table_name=table_name,
            logger=logger
        )

        logger.info("ETL-процесс успешно завершён!")
        return True

    except Exception as err:
        logger.error(f"Критическая ошибка в ETL-процессе: {err}")
        return False


In [10]:
# Формируем запросы для агрегации

# Запрос1. Количество уникальных пользователей за день
# Query 1: Unique users by day

query1 = """
SELECT DATE(created_at) as date,
COUNT(distinct lti_user_id) as unique_users_by_day
FROM api_data
GROUP BY date
ORDER BY date ASC
"""

# Запрос2. Максимальная активность пользователей по дням
# Query 2: Max activity by day

query2 = """
SELECT COUNT (attempt_type) as total_attempts,
DATE(created_at) as date
FROM api_data
GROUP BY date
ORDER BY total_attempts DESC
"""
# Запрос3. Общее количество попыток для каждого пользователя
# Query 3: Total attempts by user

query3 = """
SELECT lti_user_id AS user,
COUNT(*) AS attempts_count
FROM api_data
GROUP BY lti_user_id
ORDER BY attempts_count DESC
"""

In [ ]:
!pip install gspread google-auth google-auth-oauthlib google-auth-httplib2

In [ ]:
!pip install oauth2client

In [ ]:
!pip install google-api-python-client


In [12]:
# ETL-процесс №2: от агрегации данных в PostgreSQL до загрузки в Google Sheets

import time
import logging
import psycopg2

# Шаг 1: Выполнение всех запросов для агрегации данных
def execute_aggregation_queries(database, user, password, host, port, logger=None):

    """Выполнение агрегационных запросов к PostgreSQL и сохранение результатов в словаре
    args:
        database: имя базы данных
        user: имя пользователя
        password: пароль
        host: хост базы данных
        port: порт базы данных
        logger: экземпляр логгера
    """

    if logger is None:
        logger = logging.getLogger(__name__)


    # Список запросов для агрегации данных
    queries = [
        ("Unique users by day", query1),
        ("Max activity", query2),
        ("Attempts by user", query3)
    ]
    # Результаты всех запросов будут храниться в словаре
    results = {}

    start_time = time.time()

    logger.info("Запуск выполнения агрегационных запросов")

    # Счётчики для успешных и неудачных запросов
    successful = 0
    failed = 0

    # Выполнение каждого запроса и сохранение результатов
    for query_name, query in queries:
        start_time = time.time()
        logger.info(f"Начало запроса '{query_name}'")

    # Подключение к базе данных
        try:
            with psycopg2.connect(
                database=database,
                user=user,
                password=password,
                host=host,
                port=port
            ) as conn:
                # Создание курсора и выполнение запроса
                with conn.cursor() as cursor:
                    cursor.execute(query)

                    if cursor.description:
                        rows = cursor.fetchall()
                        columns = [desc[0] for desc in cursor.description]

                        #  конвертация для Google Sheets (например, даты в строки)
                        data_converted = [
                            [
                                v.isoformat() if hasattr(v, "isoformat") else v
                                for v in row
                            ]
                            for row in rows
                        ]
                        # Сохранение результатов в словаре
                        results[query_name] = {
                            "columns": columns,
                            "data": data_converted
                        }

                        logger.info(
                            f"'{query_name}' выполнен: {len(data_converted)} строк "
                            f"за {time.time() - start_time:.2f} сек"
                        )
                        successful += 1
                    # Если запрос не возвращает данных (например, DDL или пустой результат)
                    else:
                        results[query_name] = {"columns": [], "data": []}
                        logger.info(f"'{query_name}' выполнен без данных")
                        successful += 1

        except Exception as err:
            logger.error(
                f"Ошибка в '{query_name}': {err}",
                exc_info=True
            )
            failed += 1

    end_time = time.time()
    duration = end_time - start_time
    logger.info(f"Итог: {successful} успешно, {failed} ошибок")
    logger.info(f"Общее время выполнения: {duration:.2f} сек")

    return results

# Шаг 2: Подключение к Google Sheets

import gspread
from google.oauth2 import service_account
import logging

def setup_google_sheets(
    credentials_file='credentials.json',
    spreadsheet_name='Aggregated Data',
    scopes=None,
    logger=None
):
    
    """Настройка подключения к Google Sheets API с использованием сервисного аккаунта
    args:
        credentials_file: путь к файлу с учетными данными сервисного аккаунта
        spreadsheet_name: имя таблицы для создания или открытия
        scopes: список прав доступа
        logger: экземпляр логгера
    """

   # Логирование для Google Sheets
    if logger is None:
        logger = logging.getLogger(__name__)


    logger.info("Начало работы с Google Sheets API")

    # Путь к файлу credentials.json и необходимые права доступа для работы с Google Sheets и Drive
    if scopes is None:
        scopes = [
            'https://www.googleapis.com/auth/spreadsheets',
            'https://www.googleapis.com/auth/drive'
        ]

    try:
        # Загрузка учетных данных 
        creds = service_account.Credentials.from_service_account_file(
            credentials_file, scopes=scopes
        )
        # Авторизация клиента gspread
        client = gspread.authorize(creds)
        logger.info("Успешная аутентификация и авторизация!")
    except Exception as err:
        logger.error(f"Ошибка при аутентификации и авторизации: {err}")
        return None

    try:
        # Пытаемся открыть существующую таблицу
        spreadsheet = client.open(spreadsheet_name)
        logger.info(f"Таблица '{spreadsheet_name}' успешно открыта!")
        return spreadsheet
    except gspread.SpreadsheetNotFound:
        # Если таблицы нет, создаём новую
        spreadsheet = client.create(spreadsheet_name)
        logger.info(f"Таблица '{spreadsheet_name}' успешно создана!")
        return spreadsheet
    except Exception as err:
        logger.error(f"Критическая ошибка при работе с таблицей: {err}")
        return None

# Шаг 3: Загрузка данных в таблицу Google Sheets

def load_data_to_google_sheets(spreadsheet=None, results=None, logger=None):

    """Загрузка результатов агрегационных запросов в Google Sheets с логированием
    args:
        spreadsheet: объект таблицы Google Sheets
        results: словарь с результатами запросов
        logger: экземпляр логгера для записи ошибок и информации
    """

    if logger is None:
        logger = logging.getLogger(__name__)

    if spreadsheet is None:
        logger.error("Объект таблицы Google Sheets не передан!")
        return False

    if results is None:
        logger.error("Данные для загрузки не переданы!")
        return False

    queries = [
        "Unique users by day",
        "Max activity",
        "Attempts by user"
    ]
    
    start_time = time.time()

    logger.info("Начало загрузки данных в Google Sheets")
    
    for query_name in queries:
        if query_name not in results:
            logger.warning(f"Запрос '{query_name}' отсутствует в результатах")
            continue
        # заголовки и данные для загрузки
        try:
            query_data = results[query_name]
            columns = query_data['columns']
            data = query_data['data']

            sheet_name = query_name.replace(' ', '_')

            # проверка листа
            try:
                worksheet = spreadsheet.worksheet(sheet_name)
                worksheet.clear()
                logger.info(f"Лист '{sheet_name}' очищен")
            except:
                worksheet = spreadsheet.add_worksheet(title=sheet_name, rows="1000", cols="20")
                logger.info(f"Лист '{sheet_name}' создан")

            # заголовки
            if columns:
                worksheet.append_row(columns)

            # данные
            if data:
                worksheet.append_rows(data)
                end_time = time.time()
                duration = end_time - start_time
                logger.info(f"{query_name}: загружено {len(data)} строк")
                logger.info(f"Время загрузки: {duration:.2f} сек")
            else:
                logger.warning(f"{query_name}: нет данных")

        except Exception as err:
            logger.error(f"Ошибка при загрузке '{query_name}': {err}", exc_info=True)

    return True

In [13]:
# Запуск ETL-процесса №2: от агрегации данных в PostgreSQL до загрузки в Google Sheets (с отправкой уведомления по почте)

def run_process_google(database, user, password, host, port, logger=None):

    """Запуск полного процесса: от агрегации данных в PostgreSQL до загрузки в Google Sheets с логированием и отправкой уведомления по почте
    args:
        database: имя базы данных
        user: имя пользователя
        password: пароль
        host: хост базы данных
        port: порт базы данных
        logger: экземпляр логгера
    """


    start_time = time.time()

    logger.info("Запуск полного процесса: от агрегации данных до загрузки в Google Sheets")

    # Шаг 1: Выполнение агрегационных запросов
    results = execute_aggregation_queries(
        database=database,
        user=user,
        password=password,
        host=host,
        port=port,
        logger=logger
    )

    if not results:
        logger.warning("Нет результатов для загрузки в Google Sheets.")
        return False

    # Шаг 2: Подключение к Google Sheets
    spreadsheet = setup_google_sheets(
        credentials_file='credentials.json',
        spreadsheet_name='Aggregated Data',
        scopes=None,
        logger=logger
    )

    if not spreadsheet:
        logger.error("Не удалось подключиться к Google Sheets.")
        return False

    # Шаг 3: Загрузка данных в Google Sheets
    success = load_data_to_google_sheets(
        spreadsheet=spreadsheet,
        results=results,
        logger=logger
    )

    if success:
        end_time = time.time()
        duration = end_time - start_time
        logger.info("Полный процесс успешно завершён!")
        logger.info(f"Общее время выполнения: {duration:.2f} сек")

        send_email(
            subject="Поздравляем!!! ETL-процесс завершён успешно!",
            body=f"Полный процесс от агрегации данных до загрузки в Google Sheets успешно завершён! Время выполнения: {duration:.2f} сек.",
            logger=logger
        )
    else:
        logger.error("Процесс завершён с ошибками при загрузке в Google Sheets.")
        send_email(
            subject="ETL-процесс завершён с ошибками",
            body="Процесс от агрегации данных до загрузки в Google Sheets завершён с ошибками. Пожалуйста, проверьте логи для деталей.",
            logger=logger
        )

    return success



In [14]:
# Настройки для Mail.ru

SMTP_SERVER = "smtp.mail.ru"
SMTP_PORT=465
SENDER_EMAIL = "tanya.zhelnova.28@mail.ru"
EMAIL_PASSWORD = "ErjUZAosoSg3vHlcpkws"
RECEIVER_EMAIL = "tzhelnova28@gmail.com"

log_file_path = 'pipeline.log'

In [15]:
import smtplib
import ssl
from email.message import EmailMessage

# Создание письма
msg = EmailMessage()
msg.set_content("Поздравляем!!! ETL-процесс завершён успешно! Полный процесс от агрегации данных до загрузки в таблицу Google Sheets успешно завершён! Пожалуйста, проверьте таблицу для просмотра результатов.")
msg["Subject"] = "Поздравляем!!! ETL-процесс завершён успешно!"
msg["From"] = "tanya.zhelnova.28@mail.ru"
msg["To"] = "tzhelnova28@gmail.com"

In [16]:
# Вариант № 1: Отправка письма через SMTP-сервер Mail.ru с уведомлением об успешном завершении ETL-процесса

import smtplib
import ssl
from email.message import EmailMessage
import logging

def send_email(
    subject,
    body,
    sender_email=SENDER_EMAIL,
    receiver_email=RECEIVER_EMAIL,
    smtp_server=SMTP_SERVER,
    smtp_port=SMTP_PORT,
    email_password=EMAIL_PASSWORD,
    logger=None
):
    """
    Отправка письма с помощью SMTP-сервера Mail.ru с обработкой ошибок и логированием.

    args:
        subject: тема письма
        body: текст письма
        sender_email: адрес отправителя
        receiver_email: адрес получателя
        smtp_server: сервер SMTP
        smtp_port: порт SMTP
        email_password: пароль электронной почты
        logger: экземпляр логгера (если None, используется __name__)
    Returns:
        bool: True при успешной отправке, False при ошибке
    """
    # Инициализация логгера
    if logger is None:
        logger = logging.getLogger(__name__)

    try:
        # Логируем начало отправки
        logger.info(
            f"Начало отправки письма. Тема: '{subject}', "
            f"Получатель: {receiver_email}"
        )

        # Создание письма
        msg = EmailMessage()
        msg.set_content(body)
        msg["Subject"] = subject
        msg["From"] = sender_email
        msg["To"] = receiver_email

        # Установка защищённого соединения и отправка письма
        context = ssl.create_default_context()
        with smtplib.SMTP_SSL(smtp_server, smtp_port, context=context) as server:
            server.login(sender_email, email_password)
            server.send_message(msg)

        # Логируем успешную отправку
        success_msg = (
            f"Письмо успешно отправлено. Тема: '{subject}', "
            f"Получатель: {receiver_email}"
        )
        logger.info(success_msg)
        print(success_msg)  # дублируем в консоль
        return True

    except smtplib.SMTPAuthenticationError as err:
        error_msg = f"Ошибка аутентификации SMTP: {err}"
        logger.error(error_msg)
        print(error_msg)
        return False

    except smtplib.SMTPException as err:
        error_msg = f"Ошибка SMTP: {err}"
        logger.error(error_msg)
        print(error_msg)
        return False

    except ssl.SSLError as err:
        error_msg = f"Ошибка SSL-соединения: {err}"
        logger.error(error_msg)
        print(error_msg)
        return False

    except Exception as err:
        error_msg = f"Неожиданная ошибка при отправке письма: {err}"
        logger.critical(error_msg, exc_info=True)  # записываем стектрейс
        print(error_msg)
        return False


# Отправляем письмо после завершения ETL-процесса
# send_email("Уведомление о процессе ETL", "Поздравляем!!! Ваш ETL-процесс завершён))) Проверьте результаты в Google Sheets.")


In [ ]:
logger = setup_logger()
send_email("Уведомление о процессе ETL", "Поздравляем!!! Ваш ETL-процесс завершён))) Проверьте результаты в Google Sheets.")

In [ ]:

# Запуск ETL-процесса № 1: от получения данных из API до загрузки в PostgreSQL

logger = setup_logger()
run_etl(api_url, params, database, user, password, host, port, table_name='api_data', logger=logger)

In [ ]:
# Запуск ETL-процесса № 2: от агрегации данных в PostgreSQL до загрузки в Google Sheets (с отправкой уведомления по почте)

logger = setup_logger()
run_process_google(database, user, password, host, port, logger=logger)

In [ ]:
# Вариант № 2: Отправка уведомления на указанный email с содержимым лог‑файла после завершения ETL-процесса


import smtplib
import ssl
from email.message import EmailMessage
import os
from datetime import datetime


def send_log_email(log_dir='logs', receiver_email=RECEIVER_EMAIL, logger=None):

    """
    Отправка письма с содержимым лог‑файла за текущий день после завершения ETL-процесса
    args:       log_dir: директория, где хранятся лог‑файлы
                receiver_email: адрес электронной почты получателя
                logger: экземпляр логгера
    """
  
    if logger is None:
        logger = logging.getLogger(__name__)

    # Формируем имя файла с текущей датой
    current_date = datetime.now().strftime('%Y-%m-%d')
    log_file_path = os.path.join(log_dir, f'pipeline_{current_date}.log')

    logger.info(f"Попытка отправить лог-файл: {log_file_path}")

    # Читаем содержимое лог‑файла
    try:
        if not os.path.exists(log_file_path):
            log_content = f"Лог‑файл {log_file_path} не найден (возможно, процесс ещё не создал его)."
            logger.warning(log_content)
        else:
            with open(log_file_path, 'r', encoding='utf-8') as file:
                log_content = file.read()
                if not log_content:
                    log_content = "Лог-файл найден, но он пуст."
                    logger.warning("Лог-файл пуст")
    except Exception as err:
        log_content = f"Критическая ошибка при чтении лог‑файла: {err}"
        logger.error(log_content)
        return False

    # Настройки для Mail.ru
    smtp_server = SMTP_SERVER
    smtp_port = SMTP_PORT
    sender_email = SENDER_EMAIL
    email_password = EMAIL_PASSWORD

    # Создаём письмо
    msg = EmailMessage()
    msg.set_content(f"Содержимое лог‑файла за {current_date}:\n\n{log_content}")
    msg["Subject"] = f"Уведомление о процессе ETL — {current_date}"
    msg["From"] = sender_email
    msg["To"] = receiver_email

    # Отправка письма
    context = ssl.create_default_context()
    try:
        with smtplib.SMTP_SSL(smtp_server, smtp_port, context=context) as server:
            server.login(sender_email, email_password)
            server.send_message(msg)
            success_msg = f"Лог за {current_date} успешно отправлен на {receiver_email}!"
            logger.info(success_msg)
            print(success_msg)
            return True
    except Exception as err:
        error_msg = f"Ошибка при отправке лог‑файла: {err}"
        logger.error(error_msg)
        print(error_msg)
        return False


In [ ]:
# Отправляем лог‑файл после завершения ETL-процесса

logger = setup_logger()
send_log_email(log_dir='logs', receiver_email=RECEIVER_EMAIL, logger=logger)